# Lab 23.2: A Test Drive for Explanation Tools

Chapter 23, Choosing, Measuring and Using XAI Tools. Run the cells in order; nothing needs to be edited. The run takes about four minutes on a laptop CPU.

The question is not "which method is best?" (Chapter 20) but "does this tool, configured this way, do what it claims on our model?" We train a gradient-boosted credit model on the German Credit data, pick the 20 applicants with the highest predicted risk (the declines), and measure five tools on them: correctness against an exact reference, stability over random seeds, cost, and agreement with one another.

Tools under test (versions pinned in the README): shap (TreeExplainer, KernelExplainer, PermutationExplainer), lime, hermai (disclosure: maintained by the book's author; tested exactly like the others) and an Explainable Boosting Machine from interpret as a glass-box alternative.

In [1]:
import time, json, itertools, warnings
import numpy as np, pandas as pd
warnings.filterwarnings("ignore")
from sklearn.datasets import fetch_openml
from sklearn.model_selection import train_test_split
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.metrics import roc_auc_score
import sklearn, shap, lime, lime.lime_tabular, interpret
from importlib.metadata import version
for p in ["numpy", "pandas", "scikit-learn", "shap", "lime", "interpret-core", "hermai"]:
    print(p, version(p))

numpy 2.2.6
pandas 2.3.3
scikit-learn 1.7.2
shap 0.49.1
lime 0.2.0.1
interpret-core 0.7.8
hermai 0.1.8


## 1. Data and model

German Credit (Statlog), 1,000 applicants, 20 attributes, label good or bad risk (UCI, CC BY 4.0), fetched from OpenML as `credit-g`. Categorical attributes are stored as integer codes so that every tool sees the same 20 columns.

In [2]:
d = fetch_openml("credit-g", version=1, as_frame=True).frame
y = (d.pop("class") == "bad").astype(int)
cats = [c for c in d.columns if str(d[c].dtype) == "category"]
X = d.copy()
for c in cats:
    X[c] = X[c].cat.codes
X = X.astype(float); cols = list(X.columns)
Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=0.3, random_state=0, stratify=y)
gbm = GradientBoostingClassifier(n_estimators=200, max_depth=3, learning_rate=0.05, random_state=0).fit(Xtr, ytr)
p = gbm.predict_proba(Xte)[:, 1]
A = Xte.iloc[np.argsort(-p)[:20]]          # the 20 highest-risk test applicants
bg = Xtr.sample(100, random_state=0)       # one background set for every Shapley estimator
print(f"test AUC {roc_auc_score(yte, p):.3f}; test applicants above 0.5 risk: {(p > 0.5).sum()} of {len(p)}")
f_logit = lambda Z: gbm.decision_function(pd.DataFrame(Z, columns=cols))
f_proba = lambda Z: gbm.predict_proba(pd.DataFrame(Z, columns=cols))

test AUC 0.778; test applicants above 0.5 risk: 57 of 300

**Checkpoint 1.** Test AUC 0.778; 57 of 300 test applicants above 0.5 risk.

## 2. Correctness: an exact reference and two estimators

For a tree ensemble, interventional TreeSHAP with a fixed background computes the Shapley values exactly (in log-odds). Kernel SHAP and Permutation SHAP estimate the same quantity by sampling. If the implementations are right, their error against TreeSHAP should shrink as the budget grows.

In [3]:
tree = shap.TreeExplainer(gbm, data=bg, feature_perturbation="interventional", model_output="raw")
t = time.time(); ref = tree.shap_values(A); t_tree = (time.time() - t) / len(A)
print("efficiency check (max |sum + base - model output|):",
      np.abs(ref.sum(1) + tree.expected_value - f_logit(A.values)).max())
print(f"mean |attribution| {np.abs(ref).mean():.3f}; TreeSHAP seconds per applicant {t_tree:.4f}")
ke = shap.KernelExplainer(f_logit, bg.values)
masker = shap.maskers.Independent(bg, max_samples=100)
shap.PermutationExplainer(f_logit, masker, seed=0)(A.values[:2], max_evals=82, silent=True)  # warm-up
conv = []
for ns, me in zip([64, 128, 256, 512, 1024, 2048, 4096], [82, 162, 322, 642, 1282, 2562, 5122]):
    ek, ep, tk, tp = [], [], [], []
    for seed in range(3):
        np.random.seed(seed); t = time.time()
        ek.append(np.abs(ke.shap_values(A.values, nsamples=ns, l1_reg=False, silent=True) - ref).mean())
        tk.append((time.time() - t) / len(A))
        t = time.time()
        ep.append(np.abs(shap.PermutationExplainer(f_logit, masker, seed=seed)(A.values, max_evals=me, silent=True).values - ref).mean())
        tp.append((time.time() - t) / len(A))
    conv.append(dict(kernel_budget=ns, kernel_error=np.mean(ek), kernel_s=np.mean(tk),
                     perm_budget=me, perm_error=np.mean(ep), perm_s=np.mean(tp)))
conv = pd.DataFrame(conv); conv

efficiency check (max |sum + base - model output|): 2.37470123565231e-08
mean |attribution| 0.206; TreeSHAP seconds per applicant 0.0016


,kernel_budget,kernel_error,kernel_s,perm_budget,perm_error,perm_s
0,64,1.129477e+10,0.009655,82,0.013559,0.011742
1,128,1.998524e-02,0.016103,162,0.010252,0.016001
2,256,1.252557e-02,0.030120,322,0.007473,0.033134
3,512,7.956769e-03,0.057919,642,0.004638,0.067334
4,1024,5.335917e-03,0.112715,1282,0.003420,0.135834
5,2048,3.760140e-03,0.221996,2562,0.002397,0.278056
6,4096,2.316908e-03,0.477708,5122,0.001686,0.602953


**Checkpoint 2.** Kernel SHAP with 64 coalitions for 20 features is ill-posed and returns errors that are orders of magnitude off; from 128 coalitions on, both estimators converge toward the exact values (mean error below 0.004 log-odds at the largest budgets, against a mean attribution of about 0.21).

In [4]:
errs, nz = [], []
for seed in range(3):
    np.random.seed(seed); k = ke.shap_values(A.values, silent=True)   # library defaults
    errs.append(np.abs(k - ref).mean()); nz.append((np.abs(k) > 0).sum(1).mean())
print(f"Kernel SHAP with defaults: mean error {np.mean(errs):.4f}; nonzero attributions per applicant {np.mean(nz):.0f} of {len(cols)}")

Kernel SHAP with defaults: mean error 0.0284; nonzero attributions per applicant 10 of 20


**Checkpoint 3.** With its defaults, KernelExplainer (shap 0.49.1) keeps only 10 of the 20 features and its mean error is about 0.028, several times the error of the same budget without feature selection. The default is documented; it is still a choice someone has to notice.

## 3. Stability over seeds, cost and agreement

Each tool explains the same 20 applicants ten times with ten seeds. For each applicant we take the set of its top-3 reasons (by absolute value) and compute the mean pairwise Jaccard overlap across the ten runs (1 means the same three reasons every time).

In [5]:
from hermai.perturbations.tabular import TabularPerturbationGenerator
from hermai.explainers import LocalExplainer
top3 = lambda v: set(np.argsort(-np.abs(np.asarray(v)))[:3])
jac = lambda a, b: len(a & b) / len(a | b)
ci = [cols.index(c) for c in cats]
gen = TabularPerturbationGenerator(categorical_features=cats); gen.fit(Xtr)
herm = LocalExplainer(gbm, gen)
runs = {k: [] for k in ["KernelSHAP default", "PermutationSHAP default", "LIME default", "HermAI default"]}
times = {k: [] for k in runs}
for seed in range(10):
    np.random.seed(seed); t = time.time()
    runs["KernelSHAP default"].append(ke.shap_values(A.values, silent=True)); times["KernelSHAP default"].append((time.time() - t) / len(A))
    t = time.time(); runs["PermutationSHAP default"].append(shap.PermutationExplainer(f_logit, masker, seed=seed)(A.values, silent=True).values)
    times["PermutationSHAP default"].append((time.time() - t) / len(A))
    le = lime.lime_tabular.LimeTabularExplainer(Xtr.values, feature_names=cols, categorical_features=ci,
                                                class_names=["good", "bad"], random_state=seed)
    t = time.time(); M = np.zeros((len(A), len(cols)))
    for i in range(len(A)):
        for j, w in le.explain_instance(A.values[i], f_proba, num_features=len(cols)).as_map()[1]:
            M[i, j] = w
    times["LIME default"].append((time.time() - t) / len(A)); runs["LIME default"].append(M)
    np.random.seed(seed); t = time.time(); M = np.zeros((len(A), len(cols)))
    for i in range(len(A)):
        fi = herm.explain(A.iloc[i]).feature_importances
        for name, w in zip(fi["feature"], fi["importance"]):
            M[i, cols.index(name)] = w
    times["HermAI default"].append((time.time() - t) / len(A)); runs["HermAI default"].append(M)
stab = {"TreeSHAP": [1.0] * len(A)}
for k, rr in runs.items():
    stab[k] = [np.mean([jac(top3(a[i]), top3(b[i])) for a, b in itertools.combinations(rr, 2)]) for i in range(len(A))]
runtime = {"TreeSHAP": t_tree, **{k: np.mean(v) for k, v in times.items()}}
pd.DataFrame({"stability (mean)": {k: np.mean(v) for k, v in stab.items()},
              "stability (worst applicant)": {k: np.min(v) for k, v in stab.items()},
              "seconds per applicant": runtime}).round(4)

,stability (mean),stability (worst applicant),seconds per applicant
TreeSHAP,1.0000,1.0000,0.0016
KernelSHAP default,0.9700,0.6778,0.2623
PermutationSHAP default,0.9750,0.7000,0.0673
LIME default,0.8533,0.4956,0.0306
HermAI default,0.8583,0.5422,0.0168


**Checkpoint 4.** Stability: TreeSHAP 1.00 by construction; Permutation SHAP and Kernel SHAP about 0.97; LIME and HermAI about 0.85, with a worst applicant near 0.5. TreeSHAP is the fastest (about 2 milliseconds per applicant); Kernel SHAP with defaults is the slowest (about a quarter of a second).

In [6]:
names = ["TreeSHAP", "KernelSHAP default", "PermutationSHAP default", "LIME default", "HermAI default"]
sets = {"TreeSHAP": [[top3(ref[i]) for i in range(len(A))]] * 10,
        **{k: [[top3(r[i]) for i in range(len(A))] for r in rr] for k, rr in runs.items()}}
agree = pd.DataFrame([[np.mean([jac(sa[i], sb[i]) for sa, sb in zip(sets[a], sets[b]) for i in range(len(A))])
                       for b in names] for a in names], index=names, columns=names)
agree.round(2)

,TreeSHAP,KernelSHAP default,PermutationSHAP default,LIME default,HermAI default
TreeSHAP,1.00,0.94,0.98,0.46,0.28
KernelSHAP default,0.94,1.00,0.93,0.47,0.27
PermutationSHAP default,0.98,0.93,1.00,0.46,0.27
LIME default,0.46,0.47,0.46,1.00,0.26
HermAI default,0.28,0.27,0.27,0.26,1.00


**Checkpoint 5.** The three Shapley tools agree closely (0.93 to 0.99). LIME shares less than half of its top-3 reasons with the exact reference on average (about 0.46), HermAI 0.1.8 about 0.28.

Why? They do not compute the same thing. LIME's tabular explainer turns each feature into a binary "same bin or category as this applicant" variable and fits a weighted linear model to probabilities; its weights answer "how does the predicted risk change when this feature leaves the applicant's bin?" HermAI 0.1.8 fits a weighted Lasso to probabilities on standardized perturbations and treats every column, including integer-coded categories, as numeric. Shapley values divide the gap between this applicant and the background average in log-odds. Disagreement here is first a sign that the questions differ; Chapter 20 discusses what to do when methods disagree.

## 4. The glass-box alternative

An Explainable Boosting Machine (EBM) is a glass-box model: its local explanation is its own additive terms, so there is nothing to estimate.

In [7]:
from interpret.glassbox import ExplainableBoostingClassifier
ebm = ExplainableBoostingClassifier(random_state=0, feature_types=["nominal" if c in cats else "continuous" for c in cols]).fit(Xtr, ytr)
print(f"EBM test AUC {roc_auc_score(yte, ebm.predict_proba(Xte)[:, 1]):.3f}")
t = time.time(); loc = ebm.explain_local(A); runtime["EBM local"] = (time.time() - t) / len(A)
e0 = loc.data(0)
print("additivity check:", abs(e0["extra"]["scores"][0] + np.sum(e0["scores"]) - ebm.decision_function(A.iloc[:1])[0]))
print(f"EBM local seconds per applicant {runtime['EBM local']:.5f}")

EBM test AUC 0.792
additivity check: 1.8041124150158794e-16
EBM local seconds per applicant 0.00018


**Checkpoint 6.** EBM test AUC 0.792 (the gradient-boosted model: 0.778; on 300 test applicants the difference is small). The local explanation adds up to the model's output exactly.

In [8]:
out = dict(stability={k: list(map(float, v)) for k, v in stab.items()}, runtime={k: float(v) for k, v in runtime.items()},
           agree=agree.values.tolist(), agree_names=names, kernel_default_error=float(np.mean(errs)))
json.dump(out, open("test_drive_results.json", "w"), indent=1)
print("saved test_drive_results.json for Lab 23.3")

saved test_drive_results.json for Lab 23.3


## Questions

1. Which of the five tools would you trust to produce the principal reasons on an adverse action notice, and what would you have to configure first?
2. The default Kernel SHAP settings drop half the features. Write the sentence you would put in a model validation report about this finding.
3. LIME and HermAI disagree with the exact Shapley values more often than not. Does that make them wrong? What question would each one answer correctly?
4. Your bank explains 10,000 declines a night. Using the runtime table, estimate the compute time for each tool.